# Imports

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import optuna
import shap
import umap
import hdbscan

from joblib import dump, load
from sklearn.metrics import (
    classification_report,
    fbeta_score,
    f1_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Constants

In [ ]:
RANDOM_SEED = 42

# Inputs

In [ ]:
DB_PATH = "../data/sqlite/data.db"
TABLE   = "network_data"

# Samples per class (defined by the `label` column).
# - int  → same cap for every class
# - None → all available data (careful: may crash on large datasets)
SAMPLES_PER_CLASS: int | None = 10_000
# Per-class overrides — take precedence over SAMPLES_PER_CLASS.
SAMPLES_PER_CLASS_OVERRIDE: dict = {}  # e.g. {"benign": 20_000}

# Binary sampling: (n_benign, n_attack) — overrides SAMPLES_PER_CLASS when set.
# Loads n_benign rows from BENIGN_LABEL and n_attack rows from all other
# classes combined, distributed evenly across attack classes.
# Set to None to fall back to SAMPLES_PER_CLASS.
BINARY_SAMPLING: tuple[int, int] | None = (400_000, 400_000)

BENIGN_LABEL: str = "benign"

# Data

## Load Data

In [ ]:
def load_dataset_from_db(
    db_path: str,
    table: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    binary_sampling: "tuple[int, int] | None" = None,
    benign_label: str = "benign",
) -> pd.DataFrame:
    """
    Load data from SQLite, sampling per class directly in the DB so that
    only the selected rows are pulled into memory.

    SQLite handles sampling with ORDER BY RANDOM() LIMIT n, avoiding the
    need to load the full dataset into pandas before filtering.

    When binary_sampling=(n_benign, n_attack) is provided, samples n_benign
    rows from benign_label and distributes n_attack evenly across all attack
    classes (n_attack // num_attack_classes per class), ignoring
    samples_per_class and override.
    """
    conn = sqlite3.connect(db_path)

    if binary_sampling is not None:
        n_benign, n_attack = binary_sampling

        q_benign = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
        df_benign = pd.read_sql(q_benign, conn, params=(benign_label, n_benign))
        print(f"  {benign_label}: {len(df_benign):,} rows")

        attack_classes = pd.read_sql(
            f'SELECT DISTINCT label FROM "{table}" WHERE label != ? AND label IS NOT NULL',
            conn, params=(benign_label,)
        )["label"].tolist()

        n_per_class = n_attack // len(attack_classes)
        attack_frames = []
        for cls in sorted(attack_classes):
            q_cls = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(q_cls, conn, params=(cls, n_per_class))
            attack_frames.append(df_cls)
            print(f"  {cls}: {len(df_cls):,} rows")

        conn.close()
        result = pd.concat([df_benign] + attack_frames, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)
        print(f"\nTotal: {len(result):,} rows")
        return result

    classes = pd.read_sql(
        f'SELECT DISTINCT label FROM "{table}" WHERE label IS NOT NULL', conn
    )["label"].tolist()
    print(f"Classes: {sorted(classes)}")

    frames = []
    for cls in classes:
        n = override.get(cls, samples_per_class)

        if n is None:
            query = f'SELECT * FROM "{table}" WHERE label = ?'
            df_cls = pd.read_sql(query, conn, params=(cls,))
        else:
            query = f'SELECT * FROM "{table}" WHERE label = ? ORDER BY RANDOM() LIMIT ?'
            df_cls = pd.read_sql(query, conn, params=(cls, n))

        frames.append(df_cls)
        print(f"  {cls}: {len(df_cls):,} rows")

    conn.close()

    result = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)
    print(f"\nTotal: {len(result):,} rows")
    return result

In [ ]:
def load_dataset_from_csv(
    data_root: str,
    dataset: str,
    samples_per_class: "int | None" = None,
    override: dict = {},
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Alternative: load from CSV files directly.
    Only use for small datasets — large ones will exhaust RAM.
    """
    import glob
    dataset_path = os.path.join(data_root, dataset)
    csv_paths = sorted(glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs found in: {dataset_path}")

    df = pd.concat((pd.read_csv(p) for p in csv_paths), ignore_index=True)

    if samples_per_class is None and not override:
        return df

    sampled = []
    for cls in df["label"].unique():
        n = override.get(cls, samples_per_class)
        cls_df = df[df["label"] == cls]
        sampled.append(cls_df.sample(min(n, len(cls_df)), random_state=random_state) if n else cls_df)

    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=random_state)

In [ ]:
df = load_dataset_from_db(
    DB_PATH,
    TABLE,
    samples_per_class=SAMPLES_PER_CLASS,
    override=SAMPLES_PER_CLASS_OVERRIDE,
    binary_sampling=BINARY_SAMPLING,
    benign_label=BENIGN_LABEL,
)
df.head()

  benign: 400,000 rows
  bruteforce: 16,397 rows
  ddos: 50,000 rows
  dos: 50,000 rows
  malware: 50,000 rows
  mitm: 50,000 rows
  recon: 50,000 rows
  spoofing: 50,000 rows
  web: 50,000 rows

Total: 766,397 rows


,src_ip,dst_ip,src_port,dst_port,protocol,timestamp,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,1.665196e+09,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,1.665223e+09,0.077334,5327.528874,25.861791,12.930895,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,1.665222e+09,0.155754,873.171296,12.840754,6.420377,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,1.665262e+09,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,1.749751e+09,0.036436,2744.532272,54.890645,27.445323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


## Data Cleaning

### Removing useless features

Some columns were discarded because they do not provide useful information
for training, or because they bias the model:
- **timestamp**: encodes the data collection time, not network behaviour.
  Including it would cause the model to memorize recording sessions rather
  than learn traffic patterns.
- **src\_ip / dst\_ip** (commented out): can be re-enabled if topological
  features are desired, but they add risk of dataset-specific overfitting.

In [ ]:
removed_features: dict[str, list[str]] = {}

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

USELESS_FEATURES = ["timestamp"]#, "src_ip", "dst_ip"]
actually_dropped = [c for c in USELESS_FEATURES if c in df.columns]
df = df.drop(columns=actually_dropped)
removed_features["useless_features"] = actually_dropped

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns)}")

Before:
  Shape: (766397, 83)
  Columns: ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'fwd_se

### Strip column-name whitespace

In [ ]:
spaces_before = [c for c in df.columns if c != c.strip()]
print("Before - columns with spaces:", spaces_before)

df.columns = df.columns.str.strip()

spaces_after = [c for c in df.columns if c != c.strip()]
print("After  - columns with spaces:", spaces_after)
df.head()

Before - columns with spaces: []
After  - columns with spaces: []


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,0.077334,5327.528874,25.861791,12.930895,12.930895,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,0.155754,873.171296,12.840754,6.420377,6.420377,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,0.036436,2744.532272,54.890645,27.445323,27.445323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Remove invalid values (NaN / Inf)

In [ ]:
numeric_df = df.select_dtypes(include="number")

print("Before:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")

df = df.replace([np.inf, -np.inf], np.nan).dropna()

numeric_df = df.select_dtypes(include="number")
print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  NaN count:  {df.isna().sum().sum()}")
print(f"  Inf count:  {np.isinf(numeric_df).sum().sum()}")
df.head()

Before:
  Shape: (766397, 82)
  NaN count:  0
  Inf count:  0

After:
  Shape: (766397, 82)
  NaN count:  0
  Inf count:  0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,0.077334,5327.528874,25.861791,12.930895,12.930895,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,0.155754,873.171296,12.840754,6.420377,6.420377,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,0.036436,2744.532272,54.890645,27.445323,27.445323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Remove duplicate rows

In [ ]:
print("Before:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

df = df.drop_duplicates()

print("\nAfter:")
print(f"  Shape: {df.shape}")
print(f"  Duplicate rows: {df.duplicated().sum()}")
df.head()

Before:
  Shape: (766397, 82)
  Duplicate rows: 25087

After:
  Shape: (741310, 82)
  Duplicate rows: 0


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,active_max,active_min,active_mean,active_std,idle_max,idle_min,idle_mean,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,0.077334,5327.528874,25.861791,12.930895,12.930895,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,0.155754,873.171296,12.840754,6.420377,6.420377,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,0.036436,2744.532272,54.890645,27.445323,27.445323,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Remove highly correlated features

Features with pairwise Pearson correlation > `CORR_THRESHOLD` add
redundant information and can slow training without improving accuracy.
We keep the first feature of each correlated pair (upper-triangle scan).

In [ ]:
CORR_THRESHOLD = 0.95

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

corr_matrix = df[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
print(f"  Dropped (corr > {CORR_THRESHOLD}): {to_drop}")

df = df.drop(columns=to_drop)
removed_features["high_correlation"] = to_drop

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (741310, 82)
  Dropped (corr > 0.95): ['bwd_pkt_len_std', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'fwd_iat_tot', 'fwd_iat_max', 'psh_flag_cnt', 'ack_flag_cnt', 'pkt_size_avg', 'fwd_seg_size_avg', 'bwd_seg_size_avg', 'fwd_pkts_b_avg', 'bwd_pkts_b_avg', 'idle_max', 'idle_mean']

After:
  Shape: (741310, 68)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,active_mean,active_std,idle_min,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,0.077334,5327.528874,25.861791,12.930895,12.930895,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,0.155754,873.171296,12.840754,6.420377,6.420377,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,0.036436,2744.532272,54.890645,27.445323,27.445323,...,64240,0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


### Remove near-zero variance features

Features whose variance is below `VARIANCE_THRESHOLD` carry almost no
discriminative signal and can cause numerical instability in some learners.

In [ ]:
VARIANCE_THRESHOLD = 1e-4

numeric_cols = df.select_dtypes(include="number").columns.tolist()
print("Before:")
print(f"  Shape: {df.shape}")

variances = df[numeric_cols].var()
low_var_cols = variances[variances < VARIANCE_THRESHOLD].index.tolist()
print(f"  Dropped (var < {VARIANCE_THRESHOLD}): {low_var_cols}")

df = df.drop(columns=low_var_cols)
removed_features["near_zero_variance"] = low_var_cols

print("\nAfter:")
print(f"  Shape: {df.shape}")
df.head()

Before:
  Shape: (741310, 68)
  Dropped (var < 0.0001): ['fwd_urg_flags', 'bwd_urg_flags', 'urg_flag_cnt']

After:
  Shape: (741310, 65)


,src_ip,dst_ip,src_port,dst_port,protocol,flow_duration,flow_byts_s,flow_pkts_s,fwd_pkts_s,bwd_pkts_s,...,init_fwd_win_byts,init_bwd_win_byts,active_max,active_min,active_mean,active_std,idle_min,idle_std,cwr_flag_count,label
320978,192.168.137.187,131.107.13.100,35169,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
73590,192.168.137.47,192.168.137.1,53473,53,17,0.077334,5327.528874,25.861791,12.930895,12.930895,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
66329,192.168.137.227,8.8.8.8,59578,53,17,0.155754,873.171296,12.840754,6.420377,6.420377,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
13908,192.168.137.133,111.235.248.121,51801,123,17,0.000000,0.000000,0.000000,0.000000,0.000000,...,-1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0,benign
533318,192.168.1.103,192.168.1.50,40780,45839,6,0.036436,2744.532272,54.890645,27.445323,27.445323,...,64240,0,0.0,0.0,0.0,0.0,0.0,0.0,0,malware


In [ ]:
final_features = df.columns.tolist()
all_removed    = [col for cols in removed_features.values() for col in cols]

sep  = "=" * 60
sep2 = "-" * 60
lines = [
    sep,
    "FEATURE REPORT — DATA CLEANING SUMMARY",
    sep,
    "",
    f"Final features used ({len(final_features)}):",
]
for feat in sorted(final_features):
    lines.append(f"  + {feat}")

lines += ["", f"Excluded features ({len(all_removed)}):"]
for reason, cols in removed_features.items():
    if cols:
        lines += ["", f"  [{reason}]"]
        for col in sorted(cols):
            lines.append(f"    - {col}")

report = "\n".join(lines) + "\n"
print(report)

os.makedirs("../docs", exist_ok=True)
with open("../docs/features_report.txt", "w") as fh:
    fh.write(report)

print("Saved → ../docs/features_report.txt")

FEATURE REPORT — DATA CLEANING SUMMARY

Final features used (65):
  + active_max
  + active_mean
  + active_min
  + active_std
  + bwd_blk_rate_avg
  + bwd_byts_b_avg
  + bwd_header_len
  + bwd_iat_max
  + bwd_iat_mean
  + bwd_iat_min
  + bwd_iat_std
  + bwd_iat_tot
  + bwd_pkt_len_max
  + bwd_pkt_len_mean
  + bwd_pkt_len_min
  + bwd_pkts_s
  + bwd_psh_flags
  + cwr_flag_count
  + down_up_ratio
  + dst_ip
  + dst_port
  + ece_flag_cnt
  + fin_flag_cnt
  + flow_byts_s
  + flow_duration
  + flow_iat_max
  + flow_iat_mean
  + flow_iat_min
  + flow_iat_std
  + flow_pkts_s
  + fwd_blk_rate_avg
  + fwd_byts_b_avg
  + fwd_header_len
  + fwd_iat_mean
  + fwd_iat_min
  + fwd_iat_std
  + fwd_pkt_len_max
  + fwd_pkt_len_mean
  + fwd_pkt_len_min
  + fwd_pkt_len_std
  + fwd_pkts_s
  + fwd_psh_flags
  + idle_min
  + idle_std
  + init_bwd_win_byts
  + init_fwd_win_byts
  + label
  + pkt_len_max
  + pkt_len_mean
  + pkt_len_min
  + pkt_len_std
  + pkt_len_var
  + protocol
  + rst_flag_cnt
  + src_ip
 

# Phase 1 — Binary Classification (Benign vs Malicious)

A LightGBM binary classifier separates **benign** from **attack** traffic.
Flows predicted as attacks are forwarded to Phase 2 for further
categorisation; flows predicted as benign are dropped from the pipeline.

Hyperparameters are tuned with **Optuna** using 3-fold stratified CV.
The decision threshold is co-optimised with the model parameters so that
recall is maximised without collapsing to the trivial all-attack solution.

In [ ]:
y = df["label"].apply(lambda x: "benign" if x == "benign" else "attack")
X = df.drop(columns=df.select_dtypes(exclude="number").columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

**Priority metric: recall** — minimising false negatives is more important
than avoiding false alarms. It is preferable to flag benign traffic as
suspicious than to let an attack slip through undetected.

The Optuna objective is **F2-score** (recall weighted 4× over precision),
which avoids the degenerate solution of labelling everything as "attack".

In [ ]:
FIXED_PARAMS = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "random_state": RANDOM_SEED,
    "verbosity": -1,
    "force_row_wise": True,
    "n_jobs": 3,
}


def _suggest_lgb_params(trial: optuna.Trial) -> dict:
    """Common LightGBM hyperparameter search space shared across all phases."""
    return {
        "num_leaves":        trial.suggest_int("num_leaves", 31, 255),
        "max_depth":         trial.suggest_int("max_depth", -1, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 1.0),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 200, 800),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq":    trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_bin":           trial.suggest_int("max_bin", 128, 512),
    }


def objective(trial: optuna.Trial) -> float:
    """
    Optuna objective for Phase 1 binary classification.

    Co-optimises model hyperparameters and the decision threshold using
    F2-score, which penalises false negatives (missed attacks) more heavily.
    """
    threshold = trial.suggest_float("threshold", 0.01, 0.5)
    params = {**FIXED_PARAMS, **_suggest_lgb_params(trial)}

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        clf = lgb.LGBMClassifier(**params)
        clf.fit(X_tr, y_tr)

        probs = clf.predict_proba(X_val)
        attack_idx = list(clf.classes_).index("attack")
        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        scores.append(fbeta_score(y_val_bin, y_pred, beta=2, zero_division=0))

    return np.mean(scores)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-05-11 17:19:52,364] A new study created in memory with name: no-name-b7782fa3-2e0e-4a7c-b956-9ff891269742
Best trial: 0. Best value: 0.849945:   5%|▌         | 1/20 [00:15<04:58, 15.72s/it, 15.72/1800 seconds]

[I 2026-05-11 17:20:08,086] Trial 0 finished with value: 0.8499453746576345 and parameters: {'threshold': 0.05887198490683254, 'num_leaves': 115, 'max_depth': 8, 'min_child_samples': 33, 'min_split_gain': 0.05565081604025779, 'learning_rate': 0.03602673745188851, 'n_estimators': 223, 'subsample': 0.9707840007748614, 'subsample_freq': 1, 'colsample_bytree': 0.6387540589986189, 'reg_alpha': 7.147996745308175e-06, 'reg_lambda': 1.3308178303466483e-05, 'max_bin': 368}. Best is trial 0 with value: 0.8499453746576345.


Best trial: 1. Best value: 0.854612:  10%|█         | 2/20 [00:50<08:09, 27.17s/it, 50.90/1800 seconds]

[I 2026-05-11 17:20:43,267] Trial 1 finished with value: 0.854612434481847 and parameters: {'threshold': 0.46510674782322026, 'num_leaves': 232, 'max_depth': 16, 'min_child_samples': 99, 'min_split_gain': 0.9246890054298501, 'learning_rate': 0.005381176204450958, 'n_estimators': 273, 'subsample': 0.8715668299051778, 'subsample_freq': 4, 'colsample_bytree': 0.6258515637947368, 'reg_alpha': 0.4117685134440368, 'reg_lambda': 0.033902867832625236, 'max_bin': 195}. Best is trial 1 with value: 0.854612434481847.


Best trial: 1. Best value: 0.854612:  15%|█▌        | 3/20 [01:07<06:20, 22.40s/it, 67.64/1800 seconds]

...

[I 2026-05-11 17:30:21,242] Trial 17 finished with value: 0.8998808518054878 and parameters: {'threshold': 0.33025639838208887, 'num_leaves': 109, 'max_depth': 9, 'min_child_samples': 23, 'min_split_gain': 0.19270709990947804, 'learning_rate': 0.06418500589446856, 'n_estimators': 616, 'subsample': 0.699472892333979, 'subsample_freq': 4, 'colsample_bytree': 0.777798970997586, 'reg_alpha': 0.02510019244850047, 'reg_lambda': 0.4720115293493255, 'max_bin': 481}. Best is trial 7 with value: 0.9115009797596135.


Best trial: 7. Best value: 0.911501:  95%|█████████▌| 19/20 [11:20<00:40, 40.44s/it, 680.57/1800 seconds]

[I 2026-05-11 17:31:12,938] Trial 18 finished with value: 0.9088118172274365 and parameters: {'threshold': 0.23679330541763527, 'num_leaves': 255, 'max_depth': 15, 'min_child_samples': 46, 'min_split_gain': 0.6003943533623262, 'learning_rate': 0.03240158292105521, 'n_estimators': 456, 'subsample': 0.8696945695201012, 'subsample_freq': 6, 'colsample_bytree': 0.7148269620830922, 'reg_alpha': 0.006982612490197269, 'reg_lambda': 0.6643747917818313, 'max_bin': 416}. Best is trial 7 with value: 0.9115009797596135.


Best trial: 7. Best value: 0.911501: 100%|██████████| 20/20 [11:58<00:00, 35.94s/it, 718.81/1800 seconds]

[I 2026-05-11 17:31:51,177] Trial 19 finished with value: 0.9058307629352088 and parameters: {'threshold': 0.2352904109797874, 'num_leaves': 200, 'max_depth': 15, 'min_child_samples': 79, 'min_split_gain': 0.6283397407507356, 'learning_rate': 0.03022148113586974, 'n_estimators': 326, 'subsample': 0.9136458305792479, 'subsample_freq': 3, 'colsample_bytree': 0.7022497914301717, 'reg_alpha': 0.001959392687656851, 'reg_lambda': 0.0015921627999607457, 'max_bin': 415}. Best is trial 7 with value: 0.9115009797596135.


In [ ]:
print("Best recall:", study.best_value)
print("Best params:", study.best_params)

Best recall: 0.9115009797596135
Best params: {'threshold': 0.1927975410900863, 'num_leaves': 213, 'max_depth': 16, 'min_child_samples': 35, 'min_split_gain': 0.6981430992486972, 'learning_rate': 0.07248822518086537, 'n_estimators': 390, 'subsample': 0.6568742721909795, 'subsample_freq': 5, 'colsample_bytree': 0.9399002956990384, 'reg_alpha': 0.037465569150534776, 'reg_lambda': 0.08140306342040055, 'max_bin': 503}


In [ ]:
best = study.best_params.copy()
threshold = best.pop("threshold")

model = lgb.LGBMClassifier(**best, **FIXED_PARAMS)
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,213
,max_depth,16
,learning_rate,0.07248822518086537
,n_estimators,390
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.6981430992486972
,min_child_weight,0.001
,min_child_samples,35


In [ ]:
probs = model.predict_proba(X_test)
attack_idx = list(model.classes_).index("attack")
y_pred = np.where(probs[:, attack_idx] > threshold, "attack", "benign")

print(classification_report(y_test, y_pred))
print(f"Recall (attack): {recall_score(y_test, y_pred, pos_label='attack'):.4f}")
print(f"Recall (benign): {recall_score(y_test, y_pred, pos_label='benign'):.4f}")

              precision    recall  f1-score   support

      attack       0.81      0.94      0.87     71507
      benign       0.94      0.79      0.86     76755

    accuracy                           0.87    148262
   macro avg       0.87      0.87      0.87    148262
weighted avg       0.88      0.87      0.87    148262



In [ ]:
dump(model, "../models/binary_classifier.pkl")

['../models/binary_classifier.pkl']

# Phase 2 — Multiclass Classification

Flows flagged as attacks by Phase 1 are categorised into known attack types.
To simulate **zero-day threat detection**, half the attack classes are
relabelled as **"unknown"** during training — the model learns that some
attack traffic does not match any known category.

Flows whose maximum predicted probability falls below
`PHASE2_CONFIDENCE_THRESHOLD` are considered ambiguous and forwarded to
Phase 3 for unsupervised clustering.

## Configuration

In [ ]:
PHASE2_SAMPLES_PER_CLASS = 50_000
# Half of the 8 attack classes are relabeled as "unknown" (proxy for zero-day attacks).
# bruteforce is mandatory (fewest samples, ~16k rows). The other three complete the half.
PHASE2_UNKNOWN_CLASSES = ["bruteforce", "mitm", "spoofing", "web"]
PHASE2_CONFIDENCE_THRESHOLD = 0.6      # max_proba below this → forwarded to Phase 3

## Data

In [ ]:
X2 = (
    df.groupby("label", group_keys=False)
      .apply(lambda g: g.sample(min(len(g), PHASE2_SAMPLES_PER_CLASS), random_state=RANDOM_SEED))
)
y2 = X2["label"].replace({cls: "unknown" for cls in PHASE2_UNKNOWN_CLASSES})
X2 = X2.drop(columns=X2.select_dtypes(exclude="number").columns)

print(y2.value_counts().to_string())
print()

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=RANDOM_SEED, stratify=y2
)
print(f"Train: {X2_train.shape}, Test: {X2_test.shape}")
print(f"Classes: {sorted(y2.unique())}")

## Hyperparameter Optimisation

In [ ]:
N_CLASSES = y2_train.nunique()

FIXED_PARAMS_P2 = {
    "objective": "multiclass",
    "num_class": N_CLASSES,
    "metric": "multi_logloss",
    "boosting_type": "gbdt",
    "class_weight": "balanced",
    "random_state": RANDOM_SEED,
    "verbosity": -1,
    "force_row_wise": True,
    "n_jobs": 3,
}


def objective_p2(trial: optuna.Trial) -> float:
    """
    Optuna objective for Phase 2 multiclass classification.

    Optimises macro-averaged F1-score so that every class — including
    "unknown" (zero-day proxy) — contributes equally to the score.
    """
    params = {**FIXED_PARAMS_P2, **_suggest_lgb_params(trial)}

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    for train_idx, val_idx in cv.split(X2_train, y2_train):
        X_tr, X_val = X2_train.iloc[train_idx], X2_train.iloc[val_idx]
        y_tr, y_val = y2_train.iloc[train_idx], y2_train.iloc[val_idx]

        clf = lgb.LGBMClassifier(**params)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_val)

        scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))

    return np.mean(scores)

In [ ]:
study_p2 = optuna.create_study(direction="maximize")
study_p2.optimize(objective_p2, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-05-11 19:22:36,715] A new study created in memory with name: no-name-6d5dcba3-639c-4785-bdce-0fadffe386fa
Best trial: 0. Best value: 0.771396:   5%|▌         | 1/20 [01:23<26:34, 83.90s/it, 83.90/1800 seconds]

[I 2026-05-11 19:24:00,616] Trial 0 finished with value: 0.7713959909953202 and parameters: {'num_leaves': 131, 'max_depth': 5, 'min_child_samples': 76, 'min_split_gain': 0.583673082996767, 'learning_rate': 0.04418822227543036, 'n_estimators': 403, 'subsample': 0.9238411718878309, 'subsample_freq': 6, 'colsample_bytree': 0.8703623394697382, 'reg_alpha': 1.175897326126904e-06, 'reg_lambda': 0.05307825149951969, 'max_bin': 199}. Best is trial 0 with value: 0.7713959909953202.


Best trial: 1. Best value: 0.783036:  10%|█         | 2/20 [06:06<1:00:11, 200.62s/it, 366.22/1800 seconds]

[I 2026-05-11 19:28:42,932] Trial 1 finished with value: 0.7830363053532573 and parameters: {'num_leaves': 93, 'max_depth': 0, 'min_child_samples': 79, 'min_split_gain': 0.14259724742149127, 'learning_rate': 0.006870143573851457, 'n_estimators': 636, 'subsample': 0.690425119799529, 'subsample_freq': 5, 'colsample_bytree': 0.8365773816881703, 'reg_alpha': 0.0006826246071336595, 'reg_lambda': 1.4572743271332953e-07, 'max_bin': 442}. Best is trial 1 with value: 0.7830363053532573.


Best trial: 2. Best value: 0.794739:  15%|█▌        | 3/20 [08:43<51:10, 180.63s/it, 523.07/1800 seconds]  

...

[I 2026-05-11 19:47:32,409] Trial 11 finished with value: 0.7925621749757538 and parameters: {'num_leaves': 53, 'max_depth': 15, 'min_child_samples': 100, 'min_split_gain': 0.38771305547507184, 'learning_rate': 0.07879525540914561, 'n_estimators': 799, 'subsample': 0.8738232395204913, 'subsample_freq': 1, 'colsample_bytree': 0.7662733158258296, 'reg_alpha': 3.889224979149495, 'reg_lambda': 5.960266064987267e-05, 'max_bin': 283}. Best is trial 10 with value: 0.7990358050151652.


Best trial: 10. Best value: 0.799036:  65%|██████▌   | 13/20 [27:46<16:25, 140.77s/it, 1666.39/1800 seconds]

[I 2026-05-11 19:50:23,109] Trial 12 finished with value: 0.7966388175449483 and parameters: {'num_leaves': 31, 'max_depth': 10, 'min_child_samples': 99, 'min_split_gain': 0.380073690330349, 'learning_rate': 0.08599560411334412, 'n_estimators': 790, 'subsample': 0.8297055482286791, 'subsample_freq': 2, 'colsample_bytree': 0.7204598944358547, 'reg_alpha': 0.031698075731401196, 'reg_lambda': 0.0002759004678962348, 'max_bin': 322}. Best is trial 10 with value: 0.7990358050151652.


Best trial: 13. Best value: 0.799327:  70%|███████   | 14/20 [31:30<13:30, 135.04s/it, 1890.62/1800 seconds]

[I 2026-05-11 19:54:07,338] Trial 13 finished with value: 0.7993273564766917 and parameters: {'num_leaves': 84, 'max_depth': 16, 'min_child_samples': 61, 'min_split_gain': 0.45690740580632283, 'learning_rate': 0.03972094147767502, 'n_estimators': 699, 'subsample': 0.7947940982658335, 'subsample_freq': 2, 'colsample_bytree': 0.7138769708744533, 'reg_alpha': 2.4050901116956195e-05, 'reg_lambda': 1.4360161190259966e-05, 'max_bin': 297}. Best is trial 13 with value: 0.7993273564766917.


In [ ]:
print("Best macro F1:", study_p2.best_value)
print("Best params:", study_p2.best_params)

Best macro F1: 0.7993273564766917
Best params: {'num_leaves': 84, 'max_depth': 16, 'min_child_samples': 61, 'min_split_gain': 0.45690740580632283, 'learning_rate': 0.03972094147767502, 'n_estimators': 699, 'subsample': 0.7947940982658335, 'subsample_freq': 2, 'colsample_bytree': 0.7138769708744533, 'reg_alpha': 2.4050901116956195e-05, 'reg_lambda': 1.4360161190259966e-05, 'max_bin': 297}


## Final Model

In [ ]:
best_p2 = study_p2.best_params.copy()
model_p2 = lgb.LGBMClassifier(**best_p2, **FIXED_PARAMS_P2)
model_p2.fit(X2_train, y2_train)

,boosting_type,'gbdt'
,num_leaves,84
,max_depth,16
,learning_rate,0.03972094147767502
,n_estimators,699
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.45690740580632283
,min_child_weight,0.001
,min_child_samples,61


In [ ]:
y2_pred = model_p2.predict(X2_test)
print(classification_report(y2_test, y2_pred))

              precision    recall  f1-score   support

      benign       0.65      0.76      0.70     10000
        ddos       0.89      0.82      0.86     10000
         dos       0.83      0.89      0.86     10000
     malware       0.84      0.80      0.82     10000
        mitm       0.73      0.77      0.75     10000
       recon       0.95      0.77      0.85     10000
    spoofing       0.77      0.78      0.77     10000
     unknown       0.65      0.82      0.72      3280
         web       0.97      0.90      0.93     10000

    accuracy                           0.81     83280
   macro avg       0.81      0.81      0.81     83280
weighted avg       0.82      0.81      0.81     83280



In [ ]:
probs_p2 = model_p2.predict_proba(X2_test)
max_conf = probs_p2.max(axis=1)

known_mask = max_conf >= PHASE2_CONFIDENCE_THRESHOLD
print(f"Flows classified by Phase 2: {known_mask.sum():,} ({known_mask.mean():.1%})")
print(f"Flows forwarded to Phase 3:  {(~known_mask).sum():,} ({(~known_mask).mean():.1%})")

Flows classified by Phase 2: 63,114 (75.8%)
Flows forwarded to Phase 3:  20,166 (24.2%)


In [ ]:
dump(model_p2, "../models/multiclass_classifier.pkl")
print("Saved → ../models/multiclass_classifier.pkl")

Saved → ../models/multiclass_classifier.pkl


# Phase 3 — Unsupervised Clustering

Flows that Phase 2 could not classify with sufficient confidence are
analysed here to discover **novel attack patterns** (potential zero-day
threats). The pipeline uses two steps:

1. **UMAP** — reduces the high-dimensional feature space to a compact
   embedding that preserves local and global structure.
2. **HDBSCAN** — density-based clustering that automatically determines
   the number of clusters and labels outliers as noise (-1).

Each discovered cluster represents a group of flows with similar network
behaviour — a candidate for a new attack category.

## Configuration

In [ ]:
# UMAP dimensionality reduction
P3_UMAP_N_COMPONENTS = 10
P3_UMAP_N_NEIGHBORS  = 30
P3_UMAP_MIN_DIST     = 0.0   # tight clusters → better HDBSCAN separation

# HDBSCAN clustering
P3_HDBSCAN_MIN_CLUSTER_SIZE = 100
P3_HDBSCAN_MIN_SAMPLES      = 50

## Data

In [ ]:
# Low-confidence flows from Phase 2 (using the test split as a proxy).
# In production these would be streamed from Phase 2's inference output.
low_conf_mask = probs_p2.max(axis=1) < PHASE2_CONFIDENCE_THRESHOLD
X3 = X2_test[low_conf_mask]

print(f"Low-confidence flows: {len(X3):,} ({low_conf_mask.mean():.1%} of Phase 2 test set)")
print(f"Feature shape: {X3.shape}")

## UMAP Dimensionality Reduction

In [ ]:
reducer = umap.UMAP(
    n_components=P3_UMAP_N_COMPONENTS,
    n_neighbors=P3_UMAP_N_NEIGHBORS,
    min_dist=P3_UMAP_MIN_DIST,
    random_state=RANDOM_SEED,
)
X3_embedded = reducer.fit_transform(X3.values)

print(f"UMAP: {X3.shape} → {X3_embedded.shape}")

## HDBSCAN Clustering

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=P3_HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=P3_HDBSCAN_MIN_SAMPLES,
    prediction_data=True,
)
cluster_labels = clusterer.fit_predict(X3_embedded)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print(f"Clusters found : {n_clusters}")
print(f"Noise points   : {n_noise:,} ({n_noise / len(cluster_labels):.1%})")

## Cluster Analysis

In [ ]:
cluster_series = pd.Series(cluster_labels, index=X3.index, name="cluster")
counts = cluster_series.value_counts().sort_index()

print("Cluster  | Size")
print("-" * 22)
for label, count in counts.items():
    tag = "(noise)" if label == -1 else ""
    print(f"  {label:>5}  | {count:>7,}  {tag}")

## Visualisation

A 2-D UMAP embedding is computed separately for visualisation only —
the clustering above uses the higher-dimensional (`P3_UMAP_N_COMPONENTS`)
embedding for better density estimation.

In [ ]:
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=P3_UMAP_N_NEIGHBORS,
    min_dist=P3_UMAP_MIN_DIST,
    random_state=RANDOM_SEED,
)
X3_2d = reducer_2d.fit_transform(X3.values)

fig, ax = plt.subplots(figsize=(10, 7))
unique_labels = sorted(set(cluster_labels))
cmap = plt.cm.get_cmap("tab20", len(unique_labels))

for i, label in enumerate(unique_labels):
    mask = cluster_labels == label
    color = "lightgrey" if label == -1 else cmap(i)
    tag   = "noise" if label == -1 else f"cluster {label}"
    ax.scatter(X3_2d[mask, 0], X3_2d[mask, 1],
               c=[color], s=2, alpha=0.5, label=tag)

ax.set_title("Phase 3 — UMAP + HDBSCAN clustering of low-confidence flows")
ax.set_xlabel("UMAP dim 1")
ax.set_ylabel("UMAP dim 2")
ax.legend(markerscale=5, loc="best", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
dump(reducer,   "../models/umap_reducer.pkl")
dump(clusterer, "../models/hdbscan_clusterer.pkl")
print("Saved → ../models/umap_reducer.pkl")
print("Saved → ../models/hdbscan_clusterer.pkl")